# Pokémon Battles Prediction · Team M⁴ · Final submissions

**Team M⁴:** Matteo Schirru, Alessia Manfredi, Daniela Madonna, Margherita Guerrieri 

This notebook reproduces our three final submissions to the
[FDS: Pokémon Battles Prediction 2026](https://www.kaggle.com/competitions/fds-pokemon-battles-prediction-2026) competition.
All the code is in our GitHub repository
[guerrieri2332706/fds-pokemon-battles-M-](https://github.com/guerrieri2332706/fds-pokemon-battles-M-):
the notebook clones it, imports the functions of our module `pokemon_module.py` and only calls them.

| Output file | Model | Features | CV accuracy |
|---|---|---|---|
| `submission_lr.csv` | Logistic regression (standardised, L2) | 154 (version 6) | 67.86% |
| `submission_gb.csv` | Histogram gradient boosting | 159 (version 7) | 67.62% |
| `submission_rf.csv` | Random forest stacked on a logistic score | 296 (forest version 2) | 67.78% |

CV accuracy: mean over the same 5 stratified folds (seed 42) for every model.

## Clone the repository 


In [1]:
!git clone https://github.com/guerrieri2332706/fds-pokemon-battles-M-.git

Cloning into 'fds-pokemon-battles-M-'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 62 (delta 23), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 596.58 KiB | 10.29 MiB/s, done.
Resolving deltas: 100% (23/23), done.


## Update the System Path

In [2]:
import sys

repo_path = '/kaggle/working/fds-pokemon-battles-M-'
if repo_path not in sys.path:
    sys.path.append(repo_path)

## Import our module

In [3]:
import numpy as np
import pandas as pd
import sklearn

from pokemon_module import (
    SEED,
    # data
    load_competition_data, find_data_dir, check_data, write_submission,
    # lookup tables and features
    build_lookups, build_logistic_features, feature_columns, build_forest_features,
    # models
    logistic_plain, logistic_scaled, gradient_boosting_search,
    random_forest, random_forest_search, logistic_sparse, stacked_forest,
    # validation
    make_cv, cv_accuracy, grid_fold_scores, fold_table,
)

print(f'Python {sys.version.split()[0]} · numpy {np.__version__} · pandas {pd.__version__} · '
      f'scikit-learn {sklearn.__version__} · seed {SEED}')

Python 3.13.18 · numpy 2.1.3 · pandas 2.3.3 · scikit-learn 1.6.1 · seed 42


## Data

The competition provides 10,000 labelled battles and 5,000 test battles. For each battle we know Player 1's
full team, Player 2's lead and the first 30 turns (the battles last 30 to 80 turns); `player_won = 1` means
that Player 1 won. `check_data` verifies ids, target and submission template.

`build_lookups` builds the only tables shared across battles (species → types and base stats, the 58 frequent
moves, the 41 frequent species) **from the training set only** and without the target, plus the Gen 2 type chart
provided by the competition. Every feature function then works on one battle at a time, so applying it to the
test set cannot leak information.

In [4]:
data = load_competition_data(find_data_dir('/kaggle/input'))
check_data(data)

y = pd.Series([b['player_won'] for b in data.train], name='player_won')
test_ids = [b['battle_id'] for b in data.test]

lookups = build_lookups(data.train, data.type_chart)
print(f'Species with known types: {len(lookups.species_types)}   '
      f'Frequent moves: {len(lookups.frequent_moves)}   Frequent species: {len(lookups.frequent_species)}')

train.jsonl: 10000 battles loaded.
test.jsonl: 5000 battles loaded.
All checks passed.
Train battles: 10000   Test battles: 5000   Player 1 win rate in train: 50.0%
Turns per battle (train + test): {30: 15000}
Species with known types: 239   Frequent moves: 58   Frequent species: 41


## Validation strategy

Every model is compared on the **same 5-fold stratified cross-validation** (`StratifiedKFold`, shuffled, seed 42),
defined once in `make_cv`. The classes are balanced, so always predicting the same class scores 50%.
A change (new features or other settings) was kept only if it raised the mean accuracy and won in most folds;
the public leaderboard never guided our choices. Standardisation and stacking live inside scikit-learn estimators,
so they are refitted on the training part of each fold.

In [5]:
cv = make_cv()
for k, (train_idx, valid_idx) in enumerate(cv.split(np.zeros(len(y)), y), start=1):
    print(f'fold {k}: {len(train_idx)} training / {len(valid_idx)} validation battles, '
          f'Player 1 wins {y.iloc[valid_idx].mean():.1%} of the validation battles')

fold 1: 8000 training / 2000 validation battles, Player 1 wins 50.0% of the validation battles
fold 2: 8000 training / 2000 validation battles, Player 1 wins 50.0% of the validation battles
fold 3: 8000 training / 2000 validation battles, Player 1 wins 50.0% of the validation battles
fold 4: 8000 training / 2000 validation battles, Player 1 wins 50.0% of the validation battles
fold 5: 8000 training / 2000 validation battles, Player 1 wins 50.0% of the validation battles


## Features of the logistic regression (versions 1–7)

Each version adds one group of features to the previous one and tests one idea about what decides a battle:

| Version | Added features |
|---|---|
| v1 | pre-battle base stats (Player 1's team means, Player 2's lead) |
| v2 | HP of the two active Pokémon at turn 30 |
| v3 | Pokémon lost by Player 1 |
| v4 | Player 2's losses, status conditions of each team, stat boosts of the active Pokémon (from here the features are standardised) |
| v5 | Pokémon in good shape (at least 50% HP, no status) |
| v6 | whole-team HP and its trend from turn 20 to 30, super-effective and resisted hits (type chart), uses of 58 frequent moves |
| v7 | immunities: attacks wasted on immune Pokémon, immune matchups at turn 30 |

The table below recomputes the cross-validated accuracy of each version.
Version 7 does not help the logistic regression, so it is used only by gradient boosting.

In [6]:
lr_train = build_logistic_features(data.train, lookups)
lr_test = build_logistic_features(data.test, lookups)

history = []
for version, table in lr_train.items():
    cols = feature_columns(table)
    scaled = version not in ('v1', 'v2', 'v3')
    model = logistic_scaled() if scaled else logistic_plain()
    scores = cv_accuracy(model, table[cols], y, cv)
    if version == 'v6':
        scores_lr = scores
    history.append({'version': version, 'features': len(cols),
                    'model': 'standardised logistic' if scaled else 'logistic',
                    'CV accuracy': scores.mean(), 'std across folds': scores.std()})

display(pd.DataFrame(history).style.format({'CV accuracy': '{:.2%}', 'std across folds': '{:.2%}'})
        .hide(axis='index'))

version,features,model,CV accuracy,std across folds
v1,6,logistic,51.78%,0.80%
v2,9,logistic,53.64%,0.64%
v3,10,logistic,58.71%,1.06%
v4,25,standardised logistic,63.12%,0.54%
v5,27,standardised logistic,65.18%,0.53%
v6,154,standardised logistic,67.86%,1.02%
v7,159,standardised logistic,67.73%,0.87%


### Submission 1 · Logistic regression (version 6)

`StandardScaler` + L2 `LogisticRegression` (`max_iter=1000`, seed 42) on the 154 features of version 6,
refitted on all 10,000 labelled battles. Cross-validated accuracy 67.86%.
On the leaderboard: 68.67% public, 66.97% private.

In [7]:
cols_v6 = feature_columns(lr_train['v6'])
final_lr = logistic_scaled().fit(lr_train['v6'][cols_v6], y)   # refit on all 10,000 training battles

predictions_lr = final_lr.predict(lr_test['v6'][cols_v6])
submission_lr = write_submission(predictions_lr, test_ids, data.sample_submission, 'submission_lr.csv')
display(submission_lr.head())

submission_lr.csv created: 5000 rows, predicted Player 1 win rate 51.2%


,battle_id,player_won
0,0,1
1,1,0
2,2,0
3,3,0
4,4,1


## Gradient boosting: choosing the setting

Trees can use thresholds and interactions that a linear model cannot, so gradient boosting gets the 159 features
of version 7 (version 6 plus the immunity features). `HistGradientBoostingClassifier` (`l2_regularization=1.0`,
seed 42) is tuned by a small grid on the same folds: learning rate 0.03 or 0.06, 7 or 15 leaves per tree,
300 or 600 iterations.

In [8]:
cols_v7 = feature_columns(lr_train['v7'])
gb_search = gradient_boosting_search(cv).fit(lr_train['v7'][cols_v7], y)

grid = pd.DataFrame(gb_search.cv_results_)[
    ['param_learning_rate', 'param_max_leaf_nodes', 'param_max_iter', 'mean_test_score', 'std_test_score']
].sort_values('mean_test_score', ascending=False)
grid.columns = ['learning_rate', 'max_leaf_nodes', 'max_iter', 'CV accuracy', 'std across folds']
display(grid.style.format({'learning_rate': '{:.2f}', 'CV accuracy': '{:.2%}', 'std across folds': '{:.2%}'})
        .hide(axis='index'))

scores_gb = grid_fold_scores(gb_search)
print('Best setting:', gb_search.best_params_)

learning_rate,max_leaf_nodes,max_iter,CV accuracy,std across folds
0.03,15,300,67.62%,0.82%
0.03,7,300,67.51%,0.54%
0.06,7,300,67.45%,0.74%
0.03,15,600,67.43%,0.62%
0.06,7,600,67.38%,0.88%
0.03,7,600,67.27%,0.72%
0.06,15,600,67.09%,0.51%
0.06,15,300,67.06%,0.61%


Best setting: {'learning_rate': 0.03, 'max_iter': 300, 'max_leaf_nodes': 15}


### Submission 2 · Gradient boosting (version 7)

The best setting of the grid refitted by `GridSearchCV` on all
10,000 labelled battles. Cross-validated accuracy 67.62%, within the fold-to-fold noise of the logistic regression.
On the leaderboard: 68.00% public, 67.26% private (our best private score).

In [9]:
final_gb = gb_search.best_estimator_   # already refitted on all 10,000 training battles

predictions_gb = final_gb.predict(lr_test['v7'][cols_v7])
submission_gb = write_submission(predictions_gb, test_ids, data.sample_submission, 'submission_gb.csv')
display(submission_gb.head())

submission_gb.csv created: 5000 rows, predicted Player 1 win rate 50.7%


,battle_id,player_won
0,0,1
1,1,1
2,2,0
3,3,0
4,4,1


## 6. Random forest: features and validation

The random forest has its own features, built in two versions:

* **version 1 (47 features):** state of the battle at turn 30 (team HP, losses, active HP) plus how the battle
  was played: HP and loss differences at turns 10 and 20, damage dealt and recovered, switches, status and priority
  moves, boosts gained, turns spent asleep, paralysed, behind a substitute, seeded or confused;
* **version 2 (296 features):** A. team condition at turn 30 (sorted HP of the six Pokémon, HP bands, usable HP,
  HP weighted by base stats, statuses); B. type effectiveness of the attacks; C. 198 sparse columns with the
  uses of the 58 frequent moves and the HP of the 41 frequent species.

The cell below recomputes each step on the same folds, including a control grid around the default forest.

In [10]:
rf_train, groups = build_forest_features(data.train, lookups)
rf_test, _ = build_forest_features(data.test, lookups)
print({name: len(cols) for name, cols in groups.items()})

steps = {
    'forest · state at turn 30': cv_accuracy(random_forest(), rf_train[groups['state']], y, cv),
    'forest · version 1': cv_accuracy(random_forest(), rf_train[groups['v1']], y, cv),
    'forest · + groups A, B': cv_accuracy(random_forest(), rf_train[groups['dense']], y, cv),
    'forest · + group C (version 2)': cv_accuracy(random_forest(), rf_train[groups['v2']], y, cv),
    'logistic (C=0.1) · version 2': cv_accuracy(logistic_sparse(), rf_train[groups['v2']], y, cv),
}
display(pd.DataFrame({'CV accuracy': {k: v.mean() for k, v in steps.items()},
                      'std across folds': {k: v.std() for k, v in steps.items()}})
        .style.format('{:.2%}'))

forest_search = random_forest_search(cv).fit(rf_train[groups['v1']], y)
forest_grid = pd.DataFrame(forest_search.cv_results_)[
    ['param_max_features', 'param_min_samples_leaf', 'mean_train_score', 'mean_test_score']
].sort_values('mean_test_score', ascending=False)
forest_grid.columns = ['max_features', 'min_samples_leaf', 'training accuracy', 'CV accuracy']
display(forest_grid.style.format({'training accuracy': '{:.2%}', 'CV accuracy': '{:.2%}'}).hide(axis='index'))
spread = forest_grid['CV accuracy'].max() - forest_grid['CV accuracy'].min()
print(f'Control grid on version 1: all settings within {100 * spread:.2f} points, so the defaults are kept.')

{'state': 7, 'v1': 47, 'dense': 98, 'sparse': 198, 'v2': 296}


,CV accuracy,std across folds
forest · state at turn 30,63.79%,0.54%
forest · version 1,65.40%,0.55%
"forest · + groups A, B",65.72%,0.94%
forest · + group C (version 2),66.37%,0.47%
logistic (C=0.1) · version 2,66.89%,1.40%


max_features,min_samples_leaf,training accuracy,CV accuracy
sqrt,5,95.31%,65.84%
log2,20,76.76%,65.71%
log2,5,94.70%,65.61%
log2,1,100.00%,65.54%
sqrt,1,100.00%,65.40%
sqrt,20,77.00%,65.25%


Control grid on version 1: all settings within 0.59 points, so the defaults are kept.


### Submission 3 · Stacked random forest (version 2)

Trees split one column at a time, so the sparse columns help the forest less than a logistic regression on the same
296 features. We therefore **stack** the two: a `StackingClassifier` gives the forest the out-of-fold probability of the logistic regression as an
extra feature, together with all 296 features (`passthrough=True`). The logistic score of each training battle
comes from an inner 5-fold split, so the forest never sees a score computed on the battle itself.
Cross-validated accuracy 67.78%; the model is then refitted on all 10,000 labelled battles.
On the leaderboard: 68.67% public, 66.66% private.

In [11]:
scores_rf = cv_accuracy(stacked_forest(), rf_train[groups['v2']], y, cv)
print(f'Stacked random forest: {scores_rf.mean():.2%} (std across folds {scores_rf.std():.2%})')

final_rf = stacked_forest().fit(rf_train[groups['v2']], y)   # refit on all 10,000 training battles

predictions_rf = final_rf.predict(rf_test[groups['v2']])
submission_rf = write_submission(predictions_rf, test_ids, data.sample_submission, 'submission_rf.csv')
display(submission_rf.head())

Stacked random forest: 67.78% (std across folds 0.92%)
submission_rf.csv created: 5000 rows, predicted Player 1 win rate 52.4%


,battle_id,player_won
0,0,1
1,1,0
2,2,0
3,3,0
4,4,1


## Summary

Fold-by-fold accuracy of the three final models on the same folds, and how often their test predictions agree.

In [12]:
finals = {'logistic regression (v6)': scores_lr,
          'gradient boosting (v7)': scores_gb,
          'stacked random forest (v2)': scores_rf}
display(fold_table(finals, reference='logistic regression (v6)'))

predictions = {'logistic regression': predictions_lr, 'gradient boosting': predictions_gb,
               'stacked random forest': predictions_rf}
agreement = pd.DataFrame({a: {b: np.mean(predictions[a] == predictions[b]) for b in predictions}
                          for a in predictions})
print('Share of the 5,000 test battles on which two models make the same prediction:')
display(agreement.style.format('{:.1%}'))

,logistic regression (v6),gradient boosting (v7),stacked random forest (v2)
fold 1,68.20%,67.05%,67.85%
fold 2,67.50%,66.95%,67.55%
fold 3,67.95%,68.30%,67.15%
fold 4,69.40%,68.90%,69.50%
fold 5,66.25%,66.90%,66.85%
mean,67.86%,67.62%,67.78%
std,1.02%,0.82%,0.92%
folds better than logistic regression (v6),-,2,3


Share of the 5,000 test battles on which two models make the same prediction:


,logistic regression,gradient boosting,stacked random forest
logistic regression,100.0%,85.4%,88.5%
gradient boosting,85.4%,100.0%,85.5%
stacked random forest,88.5%,85.5%,100.0%


The three models are within 0.25 points of each other in cross-validation, less than the variation between folds
(about one point), so we submitted the best-validated model of each family. All of them plateau near 68%:
with only the first 30 turns observed, the information in the timeline, more than the model, limits the accuracy.

| Submission | CV | Public | Private |
|---|---|---|---|
| Logistic regression (v6) | 67.86% | 68.67% | 66.97% |
| Gradient boosting (v7) | 67.62% | 68.00% | 67.26% |
| Stacked random forest (v2) | 67.78% | 68.67% | 66.66% |

Every submission we made, including the untuned baseline, scored higher on the public split (1,500 battles)
than on the private one (3,500 battles); our report discusses this gap.